# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata attributes using object properties
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All references are made using the `@id` fields as specified by the Croissant schema.

In [ ]:
# List record sets
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record Set: {rs['@id']}")
    print(f"Name: {rs.get('name', 'N/A')}")
    print(f"Description: {rs.get('description', 'N/A')}")
    if 'fields' in rs:
        print("Fields:")
        for f in rs['fields']:
            print(f"  - {f['@id']}: {f.get('name', 'N/A')} ({f.get('dataType', 'N/A')})")
    else:
        print("No fields found.")
    print("---")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Example:** We will load all available record sets using their `@id` fields.

In [ ]:
# Gather all record set ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load records for each record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"DataFrame for {record_set_id} loaded with {len(df)} records.\nColumns: {df.columns.tolist()}\n---")

# Example: Show columns and preview for the first record set
if record_set_ids:
    print(f"Columns in first Record Set ({record_set_ids[0]}):")
    print(dataframes[record_set_ids[0]].columns.tolist())
    dataframes[record_set_ids[0]].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
This section includes operations such as removing outliers, transforming data distributions, or grouping data by key attributes, always referencing fields by their `@id`.

**Example:** Choose a numeric field (such as Age) from the first record set, filter for Age > 60, normalize, and group by a categorical field (such as Sex).

In [ ]:
# Example EDA: Use the first record set and its fields
rs_id = record_set_ids[0] if record_set_ids else None
df = dataframes.get(rs_id, pd.DataFrame())

# Find a numeric field (here we guess 'age' is present)
possible_numeric_fields = [c for c in df.columns if 'age' in c.lower()]  # Search for 'age' in column names
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
    threshold = 60
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field, e.g., Sex
    possible_group_fields = [c for c in df.columns if 'sex' in c.lower()]
    group_field_id = possible_group_fields[0] if possible_group_fields else None

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field matching 'age' found in columns. Please check the data dictionary.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

**Example:** Plot the distribution of Age, and a histogram grouped by Sex.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if rs_id and possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
    # Histogram of Age
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Grouped histogram by Sex if field exists
    if possible_group_fields:
        group_field_id = possible_group_fields[0]
        plt.figure(figsize=(8,4))
        sns.histplot(df, x=numeric_field_id, hue=group_field_id, bins=10, kde=True, multiple="stack")
        plt.title(f"{numeric_field_id} Distribution grouped by {group_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.legend(title=group_field_id)
        plt.show()


## 6. Conclusion
In this notebook, we've demonstrated how to load, examine, and analyze the FAIR² dataset using `mlcroissant`. We explored dataset structure by referencing entities using their `@id` fields, loaded the available record sets, filtered and normalized key numeric fields, grouped by categorical attributes, and visualized data distributions.

This process enables scalable, reproducible biomedical analysis while ensuring that FAIR principles are maintained throughout. For more advanced analysis, refer to the full Croissant schema and documentation at https://mlcommons.org/croissant/.